# Assignment 4 — Segmentation and Clustering
### Retail Sales Dataset  |  100 Marks  |  5 Tasks

---

**Prerequisite:** Run `A1_Data_Cleaning.ipynb` first.

| Assignment | Notebook |
|---|---|
| Assignment 1 | Data Cleaning |
| Assignment 2 | Exploratory Data Analysis |
| Assignment 3 | Statistical Analysis |
| Assignment 4 | Segmentation and Clustering |
| Assignment 5 | Time Series Forecasting |


## Environment Setup

In [ ]:
# Run once to install any missing packagesimport subprocess, syspkgs = ['pandas','numpy','matplotlib','seaborn','scipy','statsmodels','scikit-learn','pmdarima','prophet']for p in pkgs:subprocess.run([sys.executable,'-m','pip','install',p,'-q'], capture_output=True)print("All packages ready.")

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.ticker as mtickerimport seaborn as snsimport warningswarnings.filterwarnings('ignore')# Plot stylesns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)plt.rcParams.update({'figure.dpi':110, 'figure.figsize':(12,5),'axes.titlesize':13, 'axes.labelsize':11})print("Imports complete ")

# Assignment 4 — Segmentation and Clustering
**Marks: 100  |  Tasks: 5**

---

## Why Segment Customers?

Not all customers are equal. Some generate 10x the revenue of others. Some buy every month; others made a single purchase two years ago. Some respond to discounts; others buy only at full price.

Customer segmentation partitions the customer base into groups that are internally similar (homogeneous within clusters) and externally different (heterogeneous between clusters). This enables:

- **Marketing:** Targeted campaigns for each segment instead of one-size-fits-all
- **Pricing:** Premium pricing for high-willingness-to-pay segments
- **Retention:** Identifying at-risk customers before they churn
- **Product development:** Understanding which segments want which features
- **Resource allocation:** Prioritizing sales effort toward high-value segments

### Types of Segmentation

| Approach | Method | Based On |
|---|---|---|
| Rule-based (RFM) | Manual quintile scoring | Recency, Frequency, Monetary value |
| Distance-based clustering | K-Means | Euclidean distance in feature space |
| Hierarchy-based clustering | Agglomerative / Divisive | Linkage distance tree |
| Density-based clustering | DBSCAN | Local neighborhood density |

### Task Overview

| Task | Method | Key Output |
|---|---|---|
| 1 | RFM Analysis | Named customer segments with scores |
| 2 | Feature Engineering | Scaled feature matrix ready for ML |
| 3 | K-Means Clustering | Optimal k, cluster assignments, profiles |
| 4 | Hierarchical Clustering | Dendrogram, Ward's linkage clusters |
| 5 | DBSCAN | Density-based clusters, noise points |

### The Curse of Dimensionality

As the number of features increases, the notion of "distance" becomes less meaningful — all points become approximately equidistant from each other. This is the curse of dimensionality. In high-dimensional spaces, distance-based clustering methods (K-Means, hierarchical) lose effectiveness. Solutions:
- Feature selection: Keep only the most informative features
- PCA: Reduce to 2-10 principal components that capture most of the variance
- Feature engineering: Combine related features into single composite metrics


In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.preprocessing import StandardScalerfrom sklearn.cluster import KMeans, DBSCANfrom sklearn.decomposition import PCAfrom sklearn.metrics import silhouette_score, adjusted_rand_scorefrom scipy.cluster.hierarchy import dendrogram, linkage, fclusterfrom scipy.spatial.distance import cdistimport warningswarnings.filterwarnings('ignore')sns.set_theme(style='whitegrid', font_scale=1.05)plt.rcParams.update({'figure.dpi':110})df = pd.read_csv('retail_sales_clean.csv', parse_dates=['order_date'])for col in ['product_category','region','gender','payment_method','order_status']:df[col] = df[col].astype('category')print(f"Loaded {df.shape[0]:,} rows")

---
## Task 1 — RFM Analysis

### Theory: The RFM Framework

RFM (Recency, Frequency, Monetary) is the most widely used and practically successful customer segmentation framework in retail and e-commerce. It was developed in direct mail marketing in the 1990s and remains the gold standard for behavior-based customer segmentation.

The framework is grounded in a core empirical principle: **the best predictor of future purchasing behavior is past purchasing behavior.**

### The Three Dimensions

| Dimension | Definition | Formula | Direction |
|---|---|---|---|
| Recency (R) | How recently did this customer last purchase? | Reference date - max(order_date) | Lower days = better |
| Frequency (F) | How many times has this customer purchased? | count(order_id) per customer | Higher = better |
| Monetary (M) | How much has this customer spent in total? | sum(revenue) per customer | Higher = better |

### Why These Three Dimensions?

- **Recency** is the strongest predictor of future purchase probability. A customer who bought yesterday is far more likely to buy again than one who bought 2 years ago.
- **Frequency** measures loyalty. High-frequency customers have demonstrated commitment to the brand and are less likely to switch to competitors.
- **Monetary** value identifies economic contribution. High-M customers are worth more to retain even if they buy infrequently.

### Quintile Scoring

Each dimension is divided into 5 equal-frequency bins (quintiles), and each customer is assigned a score from 1 to 5:

- R score: 5 = most recent (lowest days since purchase), 1 = least recent
- F score: 5 = most frequent, 1 = least frequent
- M score: 5 = highest spender, 1 = lowest spender

The combined RFM score (R + F + M, range 3-15) provides a single ranking of customer value.

### Named Segments and Strategy

| Segment | Typical Profile | Strategy |
|---|---|---|
| Champions | High R, F, and M | VIP treatment; loyalty rewards; product advocacy |
| Loyal Customers | High F and M; moderate R | Early access to new products; exclusive offers |
| Potential Loyalists | Recent buyer; moderate F | Onboarding sequence; cross-sell recommendations |
| New Customers | Very high R; very low F | Welcome series; first repeat purchase incentive |
| At Risk | Previously high F/M; low R | Win-back campaign; ask for feedback |
| Cannot Lose | High historical F/M; very low R | Aggressive win-back; direct outreach |
| Lost | Low on all dimensions | Reactivation email with strong incentive; or stop spending |


In [ ]:
# Compute R, F, M per customerREF_DATE = pd.Timestamp('2025-01-01')rfm = df.groupby('customer_id').agg(recency = ('order_date', lambda x: (REF_DATE - x.max()).days),frequency = ('order_id', 'count'),monetary = ('sales_amount', 'sum')).reset_index()print("RFM Table (first 10 rows):")print(rfm.head(10).to_string(index=False))print(f"\nShape: {rfm.shape}")print(f"\nRFM Summary:")print(rfm[['recency','frequency','monetary']].describe().round(1).to_string())

In [ ]:
# Score each metric 1-5 using quintiles# Recency: LOWER days = BETTER = higher score → reverse labelsrfm['R'] = pd.qcut(rfm['recency'], q=5, labels=[5,4,3,2,1]).astype(int)rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)rfm['M'] = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)rfm['RFM_Score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)print("RFM with scores (first 10 rows):")print(rfm[['customer_id','recency','frequency','monetary','R','F','M','RFM_Score']].head(10).to_string(index=False))print(f"\nUnique RFM scores: {rfm['RFM_Score'].nunique()}")

In [ ]:
# Map scores to named segmentsdef rfm_segment(score):if score in ['555','554','545','544','535']:return 'Champions'elif score in ['543','444','435','445','543','453']:return 'Loyal Customers'elif score in ['512','513','414','415','512','511']:return 'Potential Loyalists'elif score in ['311','411','331']:return 'Promising'elif score in ['255','254','245','244','253','252']:return 'At Risk'elif score in ['155','154','145','144']:return 'Cannot Lose'elif score in ['111','112','121','131','141']:return 'Hibernating'else:return 'Others'rfm['segment'] = rfm['RFM_Score'].apply(rfm_segment)seg_summary = rfm.groupby('segment').agg(count=('customer_id','count'),avg_recency=('recency','mean'),avg_frequency=('frequency','mean'),avg_monetary=('monetary','mean'),total_value=('monetary','sum')).round(1).sort_values('total_value', ascending=False)print("Segment Summary:")print(seg_summary.to_string())

In [ ]:
# Visualise segment distributionfig, axes = plt.subplots(1, 2, figsize=(14, 5))# Bar chart: countseg_counts = rfm['segment'].value_counts()colors_seg = sns.color_palette('Set2', len(seg_counts))seg_counts.plot(kind='barh', ax=axes[0], color=colors_seg)axes[0].set_title('Customer Count per Segment')axes[0].set_xlabel('Number of Customers')# Bubble chart: Recency vs Frequency, size = Monetaryseg_agg = rfm.groupby('segment').agg(avg_recency=('recency','mean'),avg_frequency=('frequency','mean'),total_monetary=('monetary','sum')).reset_index()scatter = axes[1].scatter(seg_agg['avg_recency'], seg_agg['avg_frequency'],s=seg_agg['total_monetary']/5000,c=range(len(seg_agg)), cmap='tab10', alpha=0.7)for _, row in seg_agg.iterrows():axes[1].annotate(row['segment'], (row['avg_recency'], row['avg_frequency']),fontsize=8, ha='center')axes[1].set_xlabel('Avg Recency (days — lower is better)')axes[1].set_ylabel('Avg Frequency (orders)')axes[1].set_title('RFM Bubble Chart\n(bubble size = total monetary value)')plt.tight_layout()plt.show()

---
## Task 2 — Feature Engineering for ML Clustering

### Theory: Preparing Data for Machine Learning Clustering

Unlike RFM's rule-based scoring, machine learning clustering algorithms (K-Means, DBSCAN, hierarchical) compute distances between data points in a mathematical feature space. This requires careful preparation of the feature matrix.

### Step 1 — Select Relevant Features

Not all features should be included in clustering. Include features that:
- Are available at prediction time (no data leakage)
- Capture meaningful customer behavior
- Are not redundant with each other (high pairwise correlation)

Remove or combine features that are:
- Near-duplicates of each other (e.g., do not include both `total_revenue` and `avg_order_value` if they are derived from the same base data)
- ID columns with no predictive meaning
- Missing more than 30% of values

### Step 2 — Handle Skewed Distributions

K-Means uses Euclidean distance, which means a feature with a range of 0-10,000 will dominate a feature with a range of 0-10. Two solutions:

1. **Log transformation:** Compresses the right tail of skewed distributions. Use `np.log1p(x)` (log(1+x)) to handle zeros.
2. **Standardization:** Scales to zero mean and unit variance regardless of the original distribution.

Apply log transformation *before* standardization for severely skewed features.

### Step 3 — Feature Scaling

This is the most critical preprocessing step for distance-based algorithms.

| Scaler | Formula | Best For |
|---|---|---|
| StandardScaler | z = (x - mean) / std | When features have different scales; most common default |
| MinMaxScaler | z = (x - min) / (max - min) | When you need values in [0,1]; sensitive to outliers |
| RobustScaler | z = (x - median) / IQR | When data has many outliers |

**Always fit the scaler on training data only**, then transform both training and any new data — this prevents data leakage from the test set into the scaling parameters.

### Step 4 — Encode Categorical Variables

K-Means cannot handle text or categorical variables directly. Options:

- **One-hot encoding (get_dummies):** Creates k binary columns for a k-category variable. Good for low-cardinality (<10 categories) nominal variables.
- **Label encoding:** Maps categories to integers 0, 1, 2, ... Good for ordinal variables where the order is meaningful.
- **Target encoding:** Replaces each category with the mean of the target variable for that category. Powerful but requires careful cross-validation to avoid leakage.

### PCA for Visualization

After scaling, reduce to 2 principal components for visualization. Even if clustering is performed in the full feature space, PCA-2D plots allow visual validation of cluster quality — well-separated colored regions indicate good clustering.


In [ ]:
# Build customer-level feature tabledef safe_mode(s):m = s.mode()return m.iloc[0] if len(m) > 0 else np.nancust_features = df.groupby('customer_id').agg(total_orders = ('order_id', 'count'),total_sales = ('sales_amount', 'sum'),total_profit = ('profit', 'sum'),avg_discount = ('discount_pct', 'mean'),avg_satisfaction= ('customer_satisfaction', 'mean'),return_rate = ('return_flag', 'mean'),avg_days_ship = ('days_to_ship', 'mean'),avg_age = ('age', 'mean'),pref_category = ('product_category', safe_mode),pref_payment = ('payment_method', safe_mode),).reset_index()print(f"Customer feature table: {cust_features.shape}")print(cust_features.head(5).to_string(index=False))

In [ ]:
# One-hot encode categoricals + scale numerics# One-hot encode preferred category and paymentcust_enc = pd.get_dummies(cust_features,columns=['pref_category','pref_payment'],drop_first=True)# Fill any remaining NaNnumeric_feat_cols = [c for c in cust_enc.columns if c != 'customer_id']cust_enc[numeric_feat_cols] = cust_enc[numeric_feat_cols].fillna(cust_enc[numeric_feat_cols].median())# Scalescaler = StandardScaler()X_scaled = scaler.fit_transform(cust_enc[numeric_feat_cols])print(f"Feature matrix shape: {X_scaled.shape}")print(f"Features: {numeric_feat_cols[:10]} ... ({len(numeric_feat_cols)} total)")print(f"\nAfter scaling — mean per feature (should be ~0):")print(pd.Series(X_scaled.mean(axis=0), index=numeric_feat_cols).round(3).head(8).to_string())

---
## Task 3 — K-Means Clustering

### Theory: How K-Means Works

K-Means is the most widely used clustering algorithm. It partitions n observations into exactly k clusters by iterating two steps until convergence:

1. **Assignment step:** Assign each point to the cluster whose centroid is nearest (by Euclidean distance)
2. **Update step:** Recompute each centroid as the mean of all points currently assigned to that cluster

Convergence occurs when assignments no longer change between iterations. K-Means is guaranteed to converge, but may converge to a local minimum — not necessarily the global optimum. This is why `n_init=10` (run the algorithm 10 times with different starting centroids and keep the best result) is standard practice.

### The Objective Function

K-Means minimizes the Within-Cluster Sum of Squares (WCSS), also called inertia:

```
WCSS = sum over k [ sum over i in cluster k [ ||xi - mu_k||^2 ] ]
```

### Choosing K — The Elbow Method

WCSS always decreases as k increases (adding more clusters always reduces total within-cluster distance). The Elbow Method plots WCSS vs k and looks for the "elbow" — the point where the rate of decrease sharply slows. Beyond the elbow, adding clusters gives diminishing returns.

The elbow is often subjective. Use the Silhouette Score as a complementary metric.

### Choosing K — The Silhouette Score

For each point i, the silhouette coefficient is:

```
s(i) = (b(i) - a(i)) / max(a(i), b(i))
```

Where:
- `a(i)` = mean distance from i to all other points in the same cluster (intra-cluster cohesion)
- `b(i)` = mean distance from i to points in the nearest other cluster (inter-cluster separation)

The average silhouette score across all points ranges from -1 to +1:
- Near +1: Points are well-matched to their cluster and poorly matched to neighboring clusters
- Near 0: Points are on or near the decision boundary
- Negative: Points may be assigned to the wrong cluster

### K-Means Assumptions and Limitations

K-Means performs best when clusters are:
- **Approximately spherical** (convex shape)
- **Roughly equal in size**
- **Well-separated** in feature space

K-Means performs poorly when clusters are:
- Elongated, crescent-shaped, or ring-shaped (use DBSCAN instead)
- Very different in density (DBSCAN or GMM)
- Very different in size (hierarchical or GMM)
- Dominated by outliers (DBSCAN handles noise explicitly)

### Cluster Profiling

After assigning cluster labels, compute the mean (or median) of each feature per cluster. This answers: "What makes Cluster 2 different from Cluster 3?" Assign business-meaningful names to each cluster based on its profile.


In [ ]:
# Elbow Methodinertias, sil_scores = [], []K_range = range(2, 11)for k in K_range:km = KMeans(n_clusters=k, random_state=42, n_init=10)labels = km.fit_predict(X_scaled)inertias.append(km.inertia_)sil_scores.append(silhouette_score(X_scaled, labels, sample_size=1000, random_state=42))print(f" k={k} inertia={km.inertia_:,.0f} silhouette={sil_scores[-1]:.4f}")

In [ ]:
# Plot Elbow and Silhouettefig, axes = plt.subplots(1, 2, figsize=(13, 5))# Elbowaxes[0].plot(K_range, inertias, 'bo-', markersize=8, linewidth=2)axes[0].set_xlabel('Number of Clusters (k)')axes[0].set_ylabel('Inertia (WCSS)')axes[0].set_title('Elbow Method\nLook for the "elbow" — diminishing returns')axes[0].set_xticks(list(K_range))# Silhouettebest_k = list(K_range)[sil_scores.index(max(sil_scores))]axes[1].plot(K_range, sil_scores, 'rs-', markersize=8, linewidth=2)axes[1].axvline(best_k, color='green', linestyle='--', label=f'Best k={best_k}')axes[1].set_xlabel('Number of Clusters (k)')axes[1].set_ylabel('Silhouette Score')axes[1].set_title('Silhouette Analysis\nHigher is better (max=1.0)')axes[1].set_xticks(list(K_range))axes[1].legend()plt.suptitle('Choosing Optimal k for K-Means', fontsize=12, y=1.01)plt.tight_layout()plt.show()print(f"\nOptimal k by silhouette: {best_k} (score={max(sil_scores):.4f})")

In [ ]:
# Fit final K-MeansOPTIMAL_K = best_kkm_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)cluster_labels = km_final.fit_predict(X_scaled)cust_features['cluster'] = cluster_labelsprint(f"K-Means with k={OPTIMAL_K}")print(f"Cluster sizes:")print(cust_features['cluster'].value_counts().sort_index().to_string())# PCA to 2D for visualisationpca = PCA(n_components=2, random_state=42)X_pca = pca.fit_transform(X_scaled)explained = pca.explained_variance_ratio_print(f"\nPCA: 2 components explain {explained.sum()*100:.1f}% of variance")print(f" PC1: {explained[0]*100:.1f}% PC2: {explained[1]*100:.1f}%")

In [ ]:
# PCA scatter coloured by clusterfig, ax = plt.subplots(figsize=(10, 7))palette = sns.color_palette('tab10', OPTIMAL_K)for k in range(OPTIMAL_K):mask = cluster_labels == kax.scatter(X_pca[mask, 0], X_pca[mask, 1],c=[palette[k]], alpha=0.4, s=18, label=f'Cluster {k}')# Plot centroidscentroids_pca = pca.transform(km_final.cluster_centers_)ax.scatter(centroids_pca[:,0], centroids_pca[:,1],c='black', marker='X', s=200, zorder=5, label='Centroids')ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)')ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)')ax.set_title(f'K-Means Clusters (k={OPTIMAL_K}) — PCA 2D Projection')ax.legend(loc='upper right', fontsize=9)plt.tight_layout()plt.show()

In [ ]:
# Cluster profiling + namingprofile_cols = ['total_orders','total_sales','total_profit','avg_discount','avg_satisfaction','return_rate','avg_days_ship','avg_age']profile = cust_features.groupby('cluster')[profile_cols].mean().round(2)print("Cluster Profiles (mean values):")print(profile.to_string())# Heatmapfig, ax = plt.subplots(figsize=(12, 5))sns.heatmap(profile.T, annot=True, fmt='.1f', cmap='RdYlGn',center=0, ax=ax, linewidths=0.5,cbar_kws={'label':'Scaled Mean Value'})ax.set_title('Cluster Profile Heatmap — Mean Feature Values per Cluster')plt.tight_layout()plt.show()# Business names based on profilesprint("\n Suggested Cluster Names (review profile above to confirm):")print(" Highest total_sales + low return_rate → 'High-Value Champions'")print(" High avg_discount + low satisfaction → 'Bargain Hunters'")print(" Low frequency + medium sales → 'Occasional Buyers'")print(" Low sales + low satisfaction → 'Dormant Customers'")

---
## Task 4 — Hierarchical Clustering

### Theory: Building a Cluster Tree

Hierarchical clustering builds a complete tree (dendrogram) of cluster relationships without requiring k to be specified in advance. The two approaches are:

- **Agglomerative (bottom-up):** Start with each point as its own cluster. At each step, merge the two closest clusters. Repeat until all points are in one cluster.
- **Divisive (top-down):** Start with all points in one cluster. At each step, split the least cohesive cluster. Repeat until each point is its own cluster.

Agglomerative hierarchical clustering is far more common in practice — it is computationally efficient and provides a natural visualization through the dendrogram.

### Linkage Methods

The linkage method defines how "distance between clusters" is computed when clusters contain multiple points:

| Linkage | Distance Definition | Shape Preference | Notes |
|---|---|---|---|
| **Single** | Minimum pairwise distance | Elongated (chain-like) | Prone to "chaining" — one long string of merges |
| **Complete** | Maximum pairwise distance | Compact, spherical | Sensitive to outliers |
| **Average** | Mean of all pairwise distances | Intermediate | Good balance |
| **Ward's** | Increase in total within-cluster variance | Compact, equal-size | Most commonly recommended; similar to K-Means objective |

**Ward's linkage** is the default recommendation for most use cases. It tends to produce compact, well-separated clusters of similar size.

### Reading the Dendrogram

The y-axis (height) represents the distance at which two clusters were merged. The x-axis represents individual samples or sub-clusters.

To choose k:
1. Look for the largest vertical gap (the longest vertical line that is not crossed by any horizontal line)
2. Draw a horizontal line through this gap
3. The number of vertical lines that the horizontal line crosses is your k

### Cutting the Dendrogram

`scipy.cluster.hierarchy.fcluster()` extracts flat cluster assignments by cutting the dendrogram:

```python
# Cut to produce exactly k=4 clusters
labels = fcluster(Z, t=4, criterion='maxclust')

# Cut at a specific distance threshold
labels = fcluster(Z, t=50, criterion='distance')
```

### Comparing with K-Means

Use the **Adjusted Rand Index (ARI)** to measure agreement between two clustering solutions:
- ARI = 1: Perfect agreement
- ARI = 0: Agreement no better than random
- High ARI between K-Means and hierarchical indicates that the cluster structure is robust — both methods find it regardless of their different assumptions.


In [ ]:
# Use a subsample for speed (hierarchical is O(n²))SAMPLE_N = min(800, len(X_scaled))idx_sample = np.random.choice(len(X_scaled), SAMPLE_N, replace=False)X_hier = X_scaled[idx_sample]labels_km_sample = cluster_labels[idx_sample]# Compute linkage matrixZ = linkage(X_hier, method='ward')print(f"Linkage matrix shape: {Z.shape}")print(f"(Each row: [idx1, idx2, distance, count_merged])")print(f"\nLast 5 merges:")print(pd.DataFrame(Z[-5:], columns=['idx1','idx2','distance','count']).round(3).to_string(index=False))

In [ ]:
# Plot dendrogramfig, ax = plt.subplots(figsize=(14, 6))dendrogram(Z, ax=ax, truncate_mode='lastp', p=30, leaf_rotation=90,color_threshold=Z[-OPTIMAL_K+1, 2]) # cut at k clusters# Draw cut linecut_height = Z[-OPTIMAL_K+1, 2]ax.axhline(y=cut_height, color='red', linestyle='--', linewidth=2,label=f'Cut at height={cut_height:.2f} → {OPTIMAL_K} clusters')ax.set_title(f'Hierarchical Clustering Dendrogram (Ward linkage)\n'f'Red line = cut height for {OPTIMAL_K} clusters')ax.set_xlabel('Samples (or cluster size in brackets)')ax.set_ylabel('Ward Distance')ax.legend()plt.tight_layout()plt.show()

In [ ]:
# Extract clusters and compare with K-Meanshier_labels = fcluster(Z, t=OPTIMAL_K, criterion='maxclust') - 1 # 0-indexedari = adjusted_rand_score(labels_km_sample, hier_labels)print(f"Adjusted Rand Index (K-Means vs Hierarchical): {ari:.4f}")print(f"Interpretation: {ari:.4f} — ", end='')if ari > 0.8: print("Very high agreement — both methods find similar structure")elif ari > 0.5: print("Moderate agreement — broadly similar but some differences")elif ari > 0.2: print("Low agreement — methods find different structures")else: print("Near-random agreement — very different partitions")print("\nHierarchical cluster sizes:")unique, counts = np.unique(hier_labels, return_counts=True)for u, c in zip(unique, counts):print(f" Cluster {u}: {c} customers")

---
## Task 5 — DBSCAN Clustering

### Theory: Density-Based Clustering

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) takes a fundamentally different approach to clustering than K-Means or hierarchical methods. Instead of partitioning based on distance to centroids or merging by linkage, DBSCAN finds regions of high density (core regions) separated by regions of low density.

### Key Concepts

DBSCAN defines clusters through two parameters:
- **eps (epsilon):** The radius of the neighborhood around each point
- **min_samples:** The minimum number of points required within eps radius for a point to be a core point

Based on these parameters, every point is classified as one of:

| Point Type | Definition | Cluster Assignment |
|---|---|---|
| **Core point** | Has >= min_samples points within eps radius | Belongs to a cluster; forms the interior |
| **Border point** | Within eps of a core point, but has fewer than min_samples neighbors | Belongs to the same cluster as the nearby core point |
| **Noise point** | Neither core nor border | Labeled -1 (outlier) |

### How DBSCAN Forms Clusters

Starting from an unvisited core point, DBSCAN expands the cluster by recursively adding all directly density-reachable points. Two clusters are separate if no core point in one cluster is within eps of any core point in the other.

This allows DBSCAN to find clusters of **arbitrary shape** — crescents, rings, elongated blobs — that completely confound K-Means.

### Advantages Over K-Means

1. **No need to specify k:** The number of clusters is determined automatically by the density structure of the data
2. **Handles arbitrary shapes:** Not limited to spherical clusters
3. **Explicit noise handling:** Points that do not belong to any dense region are labeled as noise, not forced into a cluster
4. **Scale-invariant:** Changing the total number of data points does not necessarily change the cluster structure

### Choosing eps — The k-Distance Graph

The k-distance graph sorts all points by their distance to the k-th nearest neighbor (where k = min_samples). The "elbow" of this curve is a good starting point for eps:
- Points to the left of the elbow (high distances) will become noise at this eps
- Points to the right (low distances) will become core points

### Weaknesses

- **Sensitive to eps and min_samples:** The results change dramatically with different parameter values
- **Struggles with varying density:** If clusters have very different densities, no single eps works well for all of them
- **High-dimensional data:** In high dimensions, all points tend to be equidistant, making the notion of "neighborhood" less meaningful (use PCA before DBSCAN on high-dimensional data)


In [ ]:
# k-distance graph to choose epsfrom sklearn.neighbors import NearestNeighbors# Work in 2D PCA space for speedX_2d = X_pca # already computed abovek = 4nbrs = NearestNeighbors(n_neighbors=k).fit(X_2d)distances, _ = nbrs.kneighbors(X_2d)k_distances = np.sort(distances[:, k-1])[::-1]fig, ax = plt.subplots(figsize=(10, 4))ax.plot(k_distances, color='steelblue', linewidth=1.5)ax.set_xlabel('Points (sorted by distance)')ax.set_ylabel(f'Distance to {k}th Nearest Neighbour')ax.set_title('k-Distance Graph — Choose eps at the "knee"')# Find knee point approximatelyknee_idx = np.argmax(np.diff(np.diff(k_distances)))eps_suggested = k_distances[knee_idx]ax.axhline(eps_suggested, color='red', linestyle='--',label=f'Suggested eps ≈ {eps_suggested:.2f}')ax.legend()plt.tight_layout()plt.show()print(f"Suggested eps: {eps_suggested:.3f}")

In [ ]:
# Fit DBSCANdbscan = DBSCAN(eps=eps_suggested, min_samples=4)db_labels = dbscan.fit_predict(X_2d)n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)n_noise = (db_labels == -1).sum()noise_pct = n_noise / len(db_labels) * 100print(f"DBSCAN results:")print(f" Clusters found: {n_clusters}")print(f" Noise points: {n_noise} ({noise_pct:.1f}%)")print(f" Label distribution: {dict(zip(*np.unique(db_labels, return_counts=True)))}")# Plotfig, axes = plt.subplots(1, 2, figsize=(14, 5))# DBSCAN plotpalette_db = sns.color_palette('tab10', n_clusters)for lbl in sorted(set(db_labels)):mask = db_labels == lblcolor = 'lightgrey' if lbl == -1 else palette_db[lbl]marker = 'x' if lbl == -1 else 'o'label = f'Noise ({n_noise})' if lbl == -1 else f'Cluster {lbl}'axes[0].scatter(X_2d[mask,0], X_2d[mask,1], c=[color], marker=marker,alpha=0.4 if lbl==-1 else 0.6, s=15, label=label)axes[0].set_title(f'DBSCAN — {n_clusters} clusters + {n_noise} noise points')axes[0].legend(fontsize=8)# K-Means for comparisonfor k in range(OPTIMAL_K):mask = cluster_labels == kaxes[1].scatter(X_pca[mask,0], X_pca[mask,1], alpha=0.4, s=15,c=[sns.color_palette('tab10',OPTIMAL_K)[k]], label=f'K-Means {k}')axes[1].set_title(f'K-Means (k={OPTIMAL_K}) — for comparison')axes[1].legend(fontsize=8)plt.suptitle('DBSCAN vs K-Means — Same PCA 2D Space', fontsize=12, y=1.01)plt.tight_layout()plt.show()

---
## Assignment 4 — Segmentation & Clustering — Complete! Well done! Proceed to the next assignment notebook. | Assignment | Notebook File |
|---|---|
| 1. Data Cleaning | `A1_Data_Cleaning.ipynb` |
| 2. EDA | `A2_EDA.ipynb` |
| 3. Statistical Analysis | `A3_Statistical_Analysis.ipynb` |
| 4. Segmentation & Clustering | `A4_Segmentation_Clustering.ipynb` |
| 5. Time Series Forecasting | `A5_Time_Series_Forecasting.ipynb` |